In [25]:
# !pip install tqdm

In [26]:
import os
import cv2
import random
import numpy as np
from tqdm import tqdm
from pathlib import Path

In [27]:
PROJECT_ROOT = Path("..")  # since notebook is inside edge_training
RAW_IMAGE_PATH = PROJECT_ROOT / "database" / "raw_images"

EDGE_DATASET_PATH = Path("edge_dataset")

TRAIN_SPLIT = 0.8

In [28]:
for split in ["train", "val"]:
    for cls in ["good", "bad"]:
        os.makedirs(EDGE_DATASET_PATH / split / cls, exist_ok=True)

In [29]:
# Blur

def apply_blur(image):
    k = random.choice([7, 9, 11, 13])
    return cv2.GaussianBlur(image, (k, k), 0)

In [30]:
# Darken

def apply_dark(image):
    factor = random.uniform(0.3, 0.6)
    return np.clip(image * factor, 0, 255).astype(np.uint8)

In [31]:
# Glare

def apply_glare(image):
    overlay = image.copy()
    h, w, _ = image.shape
    
    x = random.randint(0, w//2)
    y = random.randint(0, h//2)
    radius = random.randint(50, 150)

    cv2.circle(overlay, (x, y), radius, (255,255,255), -1)
    
    alpha = 0.4
    return cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0)

In [32]:
all_images = list(RAW_IMAGE_PATH.glob("*.jpg"))
random.shuffle(all_images)

split_index = int(len(all_images) * TRAIN_SPLIT)

train_images = all_images[:split_index]
val_images = all_images[split_index:]

In [33]:
def process_images(image_list, split):
    for img_path in tqdm(image_list):
        img = cv2.imread(str(img_path))

        # Save GOOD image
        good_path = EDGE_DATASET_PATH / split / "good" / img_path.name
        cv2.imwrite(str(good_path), img)

        # Generate BAD image
        distortion_type = random.choice(["blur", "dark", "glare"])

        if distortion_type == "blur":
            bad_img = apply_blur(img)
        elif distortion_type == "dark":
            bad_img = apply_dark(img)
        else:
            bad_img = apply_glare(img)

        bad_filename = img_path.stem + "_bad.jpg"
        bad_path = EDGE_DATASET_PATH / split / "bad" / bad_filename
        cv2.imwrite(str(bad_path), bad_img)

In [34]:
process_images(train_images, "train")
process_images(val_images, "val")

print("Edge dataset generated successfully.")

100%|██████████| 564/564 [00:05<00:00, 98.85it/s] 

Edge dataset generated successfully.
